# Prompt en Falcon-7B-Instruct

El presente notebook emplea el modelo Falcon-7B-Instruct para indicar si el texto de entrada es positivo o negativo.

No se reentrena Falcon, se enfoca en:
* *Prompt Engineering*
* *In-Context Learning*

para realizar la clasificación sin modificar los pesos.

## Instalación de librerías

Instalar las librerías necesarias.

* **transformers**: Para los modelos y tokenizadores.
* **accelerate**: Para optimizar el entrenamiento.
* **torch**: Framework de deep learning usado por Falcon
* **sentencepiece**	Tokenizador requerido por algunos modelos

In [ ]:
# !pip install -q transformers accelerate torch sentencepiece

In [ ]:
# Importar librerías
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [ ]:
# Se indica que modelo se va a descargar desde Hugging Face
model_name = "tiiuae/falcon-7b-instruct"

In [ ]:
# Se descarga y carga automáticamente lo siguiente:
# - Vocabulario
# - Reglas de tokenización
# - Configuración textual

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# Se carga el modelo
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",          # Hace que el modelo use GPU o CPU
    torch_dtype=torch.float16   # Usa precisión de 16 bits (menos memoria, inferencia más rápida, ..)
)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

In [ ]:
# Pipeline de generación

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

In [ ]:
# Función para clasificación de sentimiento usando prompting
def clasificar_sentimiento(texto):

    prompt = f"""
    Clasifica el siguiente texto como POSITIVO o NEGATIVO.

    Reglas:
    - Responde únicamente con una palabra.
    - Las únicas respuestas válidas son:
      POSITIVO
      NEGATIVO

    Texto:
    "{texto}"

    Respuesta:
    """

    respuesta = generator(
        prompt,
        max_new_tokens=5,   # Limita la cantidad de texto generado
        do_sample=False,    # Desactiva alietoriedad
        temperature=0.0     # Reduce la creatividad
    )

    resultado = respuesta[0]["generated_text"]

    # Extraer únicamente la parte generada
    resultado = resultado.split("Respuesta:")[-1].strip()

    return resultado

In [ ]:
# Ejemplos
texto1 = "El servicio fue excelente y el producto superó mis expectativas."
texto2 = "La experiencia fue terrible y no volvería a comprar."

print("Texto 1:", clasificar_sentimiento(texto1))
print("Texto 2:", clasificar_sentimiento(texto2))

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Texto 1: "POSITIVO
Texto 2: "NEGATIV
